In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/varunika18/crime-severity/crime_severity_dataset_clean.csv
/kaggle/input/datasets/varunika18/crime-severity/synthetic_incident_severity_30k.csv


In [2]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [3]:
# ============================================
# 1. LOAD DATASET
# ============================================

df = pd.read_csv("/kaggle/input/datasets/varunika18/crime-severity/synthetic_incident_severity_30k.csv")

print("Original shape:", df.shape)
print("\nColumns:", df.columns.tolist())


Original shape: (27960, 3)

Columns: ['text', 'category', 'severity']


In [4]:
# ============================================
# 2. CLEAN DATA
# ============================================

df = df.dropna(subset=["text", "category", "severity"])

df["text"] = df["text"].astype(str).str.strip()
df["category"] = df["category"].astype(str).str.strip()
df["severity"] = df["severity"].astype(str).str.strip()

In [5]:
# Remove empty text
df = df[df["text"] != ""]

# Remove duplicate complaints
df = df.drop_duplicates(subset=["text"])

print("\nAfter cleaning:", df.shape)


After cleaning: (27949, 3)


In [6]:
# ============================================
# 3. CHECK CLASS DISTRIBUTION
# ============================================

print("\nSeverity distribution:")
print(df["severity"].value_counts())

print("\nCategory distribution:")
print(df["category"].value_counts().head(20))


Severity distribution:
severity
High        9701
Critical    9327
Normal      8921
Name: count, dtype: int64

Category distribution:
category
Infrastructure or Facility Issue                       1214
Health and Safety                                       970
Safety Hazard                                           635
Safety Hazard, Infrastructure or Facility Issue         552
Technical Issue                                         492
Academic Misconduct                                     310
Corruption or Bribery                                   302
Academic Misconduct, Abuse of Authority                 297
Infrastructure or Facility Issue, Health and Safety     295
Discrimination or Bias                                  276
Workplace Misconduct                                    275
Customer Complaint                                      270
Theft, Security Concern                                 267
Health and Safety, Safety Hazard                        266
Theft            

In [7]:
# ============================================
# 4. COMBINE TEXT + CATEGORY
# ============================================

df["combined_text"] = (
    df["text"] + " CATEGORY_" + df["category"]
)

X = df["combined_text"]
y = df["severity"]

In [8]:
# ============================================
# 5. TRAIN TEST SPLIT
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


In [9]:
# ============================================
# 6. WORD + CHARACTER TF-IDF
# ============================================

features = FeatureUnion([

    # Word-level features
    ("word", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        max_features=100000
    )),

    # Character-level features
    ("char", TfidfVectorizer(
        analyzer="char",
        ngram_range=(3, 5),
        min_df=2,
        sublinear_tf=True,
        max_features=100000
    ))
])

In [10]:
# ============================================
# 7. TRANSFORM TEXT
# ============================================

X_train_tfidf = features.fit_transform(X_train)
X_test_tfidf = features.transform(X_test)

print("\nTF-IDF shape:")
print(X_train_tfidf.shape)


TF-IDF shape:
(22359, 124109)


In [11]:
# ============================================
# 8. SVM
# ============================================

svm = LinearSVC()


# ============================================
# 9. HYPERPARAMETER TUNING
# ============================================

parameters = {
    "C": [0.1, 0.5, 1, 2, 5, 10],
    "class_weight": [None, "balanced"]
}

grid = GridSearchCV(
    svm,
    parameters,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1
)


In [12]:
# ============================================
# 10. TRAIN
# ============================================

grid.fit(X_train_tfidf, y_train)

Fitting 5 folds for each of 12 candidates, totalling 60 fits


GridSearchCV(cv=5, estimator=LinearSVC(), n_jobs=-1,
             param_grid={'C': [0.1, 0.5, 1, 2, 5, 10],
                         'class_weight': [None, 'balanced']},
             scoring='f1_macro', verbose=1)

In [13]:
# ============================================
# 11. BEST MODEL
# ============================================

model = grid.best_estimator_

print("\nBest parameters:")
print(grid.best_params_)

print("\nBest CV score:")
print(grid.best_score_)



Best parameters:
{'C': 0.5, 'class_weight': None}

Best CV score:
0.916619019140535


In [14]:
# ============================================
# 12. TEST
# ============================================

y_pred = model.predict(X_test_tfidf)

In [15]:
# ============================================
# 13. ACCURACY
# ============================================

accuracy = accuracy_score(y_test, y_pred)

print("\n================================")
print("FINAL MODEL PERFORMANCE")
print("================================")

print("Accuracy:", round(accuracy, 4))
print("Accuracy %:", round(accuracy * 100, 2))



FINAL MODEL PERFORMANCE
Accuracy: 0.9163
Accuracy %: 91.63


In [16]:
# ============================================
# 14. CLASSIFICATION REPORT
# ============================================

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_pred,
        digits=4
    )
)



Classification Report:

              precision    recall  f1-score   support

    Critical     0.9281    0.9266    0.9273      1866
        High     0.8730    0.8933    0.8831      1940
      Normal     0.9529    0.9305    0.9416      1784

    accuracy                         0.9163      5590
   macro avg     0.9180    0.9168    0.9173      5590
weighted avg     0.9169    0.9163    0.9165      5590



In [17]:
# ============================================
# 15. CONFUSION MATRIX
# ============================================

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_test,
        y_pred
    )
)



Confusion Matrix:
[[1729  134    3]
 [ 128 1733   79]
 [   6  118 1660]]


In [18]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted")
recall = recall_score(y_test, y_pred, average="weighted")
f1 = f1_score(y_test, y_pred, average="weighted")

# Create performance table
model1_results = pd.DataFrame({
    "Model": ["Linear SVM"],
    "Accuracy": [accuracy],
    "Precision": [precision],
    "Recall": [recall],
    "F1 Score": [f1]
})

# Display rounded values
model1_results[["Accuracy", "Precision", "Recall", "F1 Score"]] = \
    model1_results[["Accuracy", "Precision", "Recall", "F1 Score"]].round(4)

print("\n================================")
print("FINAL MODEL PERFORMANCE")
print("================================")
print(model1_results)


FINAL MODEL PERFORMANCE
        Model  Accuracy  Precision  Recall  F1 Score
0  Linear SVM    0.9163     0.9169  0.9163    0.9165


In [19]:
# ============================================
# 16. SAVE MODEL
# ============================================

joblib.dump(model, "crime_severity_svm.pkl")
joblib.dump(features, "crime_severity_tfidf.pkl")

print("\nModel saved successfully!")



Model saved successfully!
